# GLAAM Address Matching- All-Splink Pipeline (Exp2)

**all-Splink** matching pipeline against the LDC
commercial premises dataset using OS NGD canonical addresses.

### Pipeline summary

| Setting | Value |
|---------|-------|
| **Matching stage** | `SplinkStage` only (no deterministic stages) |
| **match_weight threshold** | 10 |
| **Canonical data** | 9 NGD CSV types (non-addressable objects excluded) |
| **Postcode strategy** | Full postcodes in address keys |
| **Pre-cleaning** | Residential filtering, OS punctuation stripping, dedup |

### Outputs

- `full_results_exp2.csv` — full matching results (all methods, all candidates)
- `exp2_results_reformatted.csv` — reformatted

## 1. Imports & configuration

In [1]:
import multiprocessing
import os
import tempfile
import time
import zipfile

import duckdb
import numpy as np
import pandas as pd
import psutil
from thefuzz import fuzz

from uk_address_matcher import AddressMatcher, SplinkStage

DATA_DIR     = 'data'
NGD_DATA_DIR = os.path.join(DATA_DIR, 'ngd')
LDC_FILE     = os.path.join(DATA_DIR, 'ldc', 'ldc_history_2025-12-31.csv')

cores = multiprocessing.cpu_count()
container_memory = os.environ.get('DUCKDB_MEMORY_LIMIT', '')
temp_dir = tempfile.gettempdir()
total_gb = psutil.virtual_memory().total // (1024 ** 3)
mem_limit = int(total_gb * 0.9)

print(f'Cores: {cores}, Memory: {container_memory or f"{mem_limit}GB"}, Temp: {temp_dir}')

Cores: 8, Memory: 9GB, Temp: /tmp


## 2. Data-loading helpers

In [2]:
_DEFAULT_COLS = [
    'uprn', 'organisationname', 'subname', 'name', 'number',
    'streetname', 'locality', 'townname',
    'primaryclassificationdescription', 'postcode',
    'fulladdress', 'latitude', 'longitude',
]
_ALT_COLS = [
    'uprn', 'alternatesubname', 'alternatename', 'alternatenumber',
    'streetname', 'locality', 'townname', 'postcode',
    'fulladdress', 'addressstatus',
]
_PSTL_COLS = [
    'uprn', 'organisationname', 'subbuildingname', 'buildingname',
    'buildingnumber', 'thoroughfare', 'dependentlocality',
    'posttown', 'postcode',
]


def _number_str(series):
    return series.astype(str).replace('<NA>', '').replace('nan', '')


def _load_ngd_csv(path):
    fname = os.path.basename(path)
    if '_altadd' in fname:
        df = pd.read_csv(path, usecols=_ALT_COLS)
        df = df.rename(columns={
            'alternatesubname': 'subname', 'alternatename': 'name',
            'alternatenumber': 'number', 'addressstatus': 'source',
        })
        df['source'] = 'alternative'
    elif '_pstladd' in fname:
        df = pd.read_csv(path, usecols=_PSTL_COLS)
        df = df.rename(columns={
            'subbuildingname': 'subname', 'buildingname': 'name',
            'buildingnumber': 'number', 'thoroughfare': 'streetname',
            'dependentlocality': 'locality', 'posttown': 'townname',
        })
        df['source'] = 'postal'
        df['number'] = pd.array(df['number'], dtype='Int64')
    else:
        df = pd.read_csv(path, usecols=_DEFAULT_COLS)
        df['source'] = 'main'
    df['number_str'] = _number_str(df['number'])
    df['type'] = fname.split('_')[2].split('.')[0]
    return df


def _join(*parts):
    return ', '.join(
        str(p).upper() for p in parts
        if pd.notna(p) and str(p) not in ('', 'nan', 'NAN')
    )


def build_canonical_key(df, fields, uid_prefix='c'):
    subset = df[['uprn'] + fields].copy()
    subset['address_c'] = [_join(*row) for row in subset[fields].itertuples(index=False)]
    result = subset[['uprn', 'address_c']].drop_duplicates().sort_values('uprn')
    result = result.reset_index(drop=True).reset_index().rename(columns={'index': f'uid_{uid_prefix}'})
    result[f'uid_{uid_prefix}'] = result[f'uid_{uid_prefix}'].astype(str)
    return result


def build_messy_key(df, fields, uid_prefix='m'):
    subset = df[['uprn', 'premises_id'] + fields].copy()
    subset['address_m'] = [_join(*row) for row in subset[fields].itertuples(index=False)]
    result = subset[['uprn', 'premises_id', 'address_m']].drop_duplicates().sort_values('premises_id')
    result = result.reset_index(drop=True).reset_index().rename(columns={'index': f'uid_{uid_prefix}'})
    result[f'uid_{uid_prefix}'] = result[f'uid_{uid_prefix}'].astype(str)
    return result


print('Data-loading helpers defined.')

Data-loading helpers defined.


## 3. Load OS NGD canonical data

In [3]:
zip_list = [
    f for f in os.listdir(NGD_DATA_DIR)
    if f.endswith('.zip') and 'streetaddress' not in f
]
for zf in zip_list:
    zip_path = os.path.join(NGD_DATA_DIR, zf)
    with zipfile.ZipFile(zip_path) as z:
        to_extract = [m for m in z.namelist() if 'rltenty.csv' not in m and 'otrclass.csv' not in m]
        for member in to_extract:
            dest = os.path.join(NGD_DATA_DIR, member)
            if not os.path.exists(dest):
                z.extract(member, NGD_DATA_DIR)
print(f'ZIPs processed: {len(zip_list)}')

csv_files = [os.path.join(NGD_DATA_DIR, f) for f in os.listdir(NGD_DATA_DIR) if f.endswith('.csv')]
print(f'Loading {len(csv_files)} CSV files...')
os_df = pd.concat([_load_ngd_csv(f) for f in csv_files], axis=0)
os_df = os_df.replace(pd.NA, np.nan)
print(f'Raw OS rows: {len(os_df):,}')

ZIPs processed: 0
Loading 9 CSV files...
Raw OS rows: 10,887,288


## 4. Clean OS data

- Filter out residential addresses
- Deduplicate
- Strip punctuation from OS fields (apostrophes, periods)

In [4]:
os_df = os_df.sort_values(['uprn', 'primaryclassificationdescription'])
os_df['primaryclassificationdescription'] = (
    os_df.groupby('uprn', sort=False)['primaryclassificationdescription'].ffill()
)
os_df = os_df[os_df['primaryclassificationdescription'] != 'Residential']
os_df['latitude']  = os_df.groupby('uprn', sort=False)['latitude'].ffill()
os_df['longitude'] = os_df.groupby('uprn', sort=False)['longitude'].ffill()
os_df = os_df.drop_duplicates(
    subset=['uprn', 'organisationname', 'subname', 'name',
            'number_str', 'streetname', 'townname', 'postcode'],
    keep='first',
)
os_df['postcode_sector'] = os_df['postcode'].str[:-2]
for _col in ['organisationname', 'subname', 'name', 'streetname']:
    os_df[_col] = os_df[_col].str.replace("'", '', regex=False).str.replace('.', '', regex=False)

print(f'Commercial rows after cleaning: {len(os_df):,}')

Commercial rows after cleaning: 1,163,429


## 5. Load LDC messy data

In [5]:
ldc_raw = pd.read_csv(LDC_FILE)
ldc_raw['address'] = (
    ldc_raw['address']
    .str.replace("'", '', regex=False)
    .str.replace('.', '', regex=False)
)
print(f'LDC raw rows: {len(ldc_raw):,}')

ldc_split = ldc_raw['address'].str.split(', ', n=10, expand=True)
for _ in range(ldc_split.shape[1]):
    no_postcode = ldc_split.index[ldc_split[ldc_split.shape[1] - 1].isna()].tolist()
    ldc_split.iloc[no_postcode] = ldc_split.iloc[no_postcode].shift(periods=1, axis=1)

n = ldc_split.shape[1]
ldc_df = ldc_split.drop(columns=[n - 3, n - 2])
ldc_df['organisationname'] = ldc_raw['tenant']
ldc_df['uprn']             = ldc_raw['uprn_id']
ldc_df['premises_id']      = ldc_raw['premises_id']
ldc_df['postcode']         = ldc_df[n - 1]
ldc_df['postcode_sector']  = ldc_df['postcode'].str[:-2]
ldc_df['streetname']       = ldc_df[n - 4]
ldc_df = ldc_df.drop_duplicates()

print(f'LDC after splitting: {len(ldc_df):,} rows')

LDC raw rows: 347,378
LDC after splitting: 280,423 rows


## 6. Build canonical & messy address keys

In [6]:
canonical_a = build_canonical_key(os_df, ['organisationname', 'streetname', 'postcode'], 'ca')
canonical_b = build_canonical_key(os_df, ['subname', 'number_str', 'streetname', 'postcode'], 'cb')
canonical_c = build_canonical_key(os_df, ['name', 'number_str', 'streetname', 'postcode'], 'cc')
canonical_d = build_canonical_key(os_df, ['name', 'subname', 'streetname', 'postcode'], 'cd')
canonical_e = build_canonical_key(os_df, ['number_str', 'streetname', 'postcode'], 'ce')
canonical_f = build_canonical_key(os_df, ['name', 'subname', 'number_str', 'streetname', 'postcode'], 'cf')

for label, df in [('A', canonical_a), ('B', canonical_b), ('C', canonical_c),
                  ('D', canonical_d), ('E', canonical_e), ('F', canonical_f)]:
    print(f'Canonical {label}: {len(df):,} rows')

Canonical A: 1,052,921 rows
Canonical B: 1,027,922 rows
Canonical C: 1,008,306 rows
Canonical D: 1,062,190 rows
Canonical E: 974,743 rows
Canonical F: 1,051,097 rows


In [7]:
_addr_token_cols = [c for c in ldc_df.columns if isinstance(c, int) and c <= n - 5]

messy_a = build_messy_key(ldc_df, ['organisationname', 'streetname', 'postcode'], 'ma')
messy_b = build_messy_key(ldc_df, _addr_token_cols + ['streetname', 'postcode'], 'mb')

print(f'Messy a: {len(messy_a):,} rows')
print(f'Messy b: {len(messy_b):,} rows')

pair_list_6 = [
    (messy_a, canonical_a, 'a'),
    (messy_b, canonical_b, 'b'),
    (messy_b, canonical_c, 'c'),
    (messy_b, canonical_d, 'd'),
    (messy_b, canonical_e, 'e'),
    (messy_b, canonical_f, 'f'),
]
print(f'Pair list: {len(pair_list_6)} rounds ready')

Messy a: 280,329 rows
Messy b: 161,796 rows
Pair list: 6 rounds ready


## 7. Matching helpers

In [8]:
def make_duckdb_con():
    c = duckdb.connect(database=':memory:')
    c.execute(f'PRAGMA threads={cores}')
    if container_memory:
        c.execute(f"PRAGMA memory_limit='{container_memory}'")
    else:
        c.execute(f"PRAGMA memory_limit='{mem_limit}GB'")
    c.execute(f"SET temp_directory='{temp_dir}'")
    return c


def run_matching_loop(pair_list, stages, label='experiment'):
    """Run the full 6-round matching loop with a given stage config."""
    import tempfile as _tmpmod
    _TMP_MESSY  = os.path.join(_tmpmod.gettempdir(), '_exp_messy.parquet')
    _TMP_CANON  = os.path.join(_tmpmod.gettempdir(), '_exp_canon.parquet')

    all_results = []
    t0 = time.time()

    for i, (df_messy_pd, df_canon_pd, method) in enumerate(pair_list):
        uid_m_col = [c for c in df_messy_pd.columns if c.startswith('uid_')][0]
        uid_c_col = [c for c in df_canon_pd.columns if c.startswith('uid_')][0]

        input_df = df_messy_pd.rename(
            columns={uid_m_col: 'unique_id', 'address_m': 'address_concat'}
        )[['unique_id', 'address_concat']].copy()
        input_df['postcode'] = input_df['address_concat'].str.rsplit(', ', n=1).str[-1]

        ref_df = df_canon_pd.rename(
            columns={uid_c_col: 'unique_id', 'address_c': 'address_concat'}
        )[['unique_id', 'address_concat']].copy()
        ref_df['postcode'] = ref_df['address_concat'].str.rsplit(', ', n=1).str[-1]

        input_df.to_parquet(_TMP_MESSY)
        ref_df.to_parquet(_TMP_CANON)

        round_con = make_duckdb_con()
        pq_input = round_con.read_parquet(_TMP_MESSY)
        pq_ref   = round_con.read_parquet(_TMP_CANON)

        print(f'  [{label}] Round {i} (method {method!r}) — '
              f'{df_messy_pd.shape[0]:,} x {df_canon_pd.shape[0]:,}')
        t_round = time.time()

        matcher = AddressMatcher(
            canonical_addresses=pq_ref,
            addresses_to_match=pq_input,
            con=round_con,
            stages=stages,
        )
        match_result = matcher.match()
        df_result = match_result.matches().df()
        df_result = df_result[df_result['resolved_canonical_id'].notna()]
        df_result = df_result.rename(columns={
            'unique_id': uid_m_col,
            'resolved_canonical_id': uid_c_col,
            'original_address_concat': 'address_m',
            'original_address_concat_canonical': 'address_c',
        })

        if df_result.empty:
            print(f'    No matches — skipping.')
            round_con.close()
            continue

        df_result['fuzz_similarity'] = df_result.apply(
            lambda r: fuzz.ratio(
                str(r.get('address_m','')).upper(),
                str(r.get('address_c','')).upper()
            ), axis=1
        )

        combined_df = df_messy_pd.merge(
            df_result[[uid_m_col, uid_c_col, 'match_weight',
                       'distinguishability', 'fuzz_similarity']],
            on=uid_m_col, how='left',
        ).merge(
            df_canon_pd, on=uid_c_col, how='left', suffixes=['_m', '_c'],
        ).sort_values(['premises_id', 'fuzz_similarity'], ascending=[True, False])
        combined_df['method'] = method
        all_results.append(combined_df)
        matched_count = combined_df[uid_c_col].notna().sum()
        round_con.close()
        print(f'    -> {matched_count:,} matched in {time.time()-t_round:.1f}s')

    elapsed = time.time() - t0
    print(f'  [{label}] All rounds done in {elapsed:.1f}s')

    if not all_results:
        return pd.DataFrame()

    full = pd.concat(all_results, ignore_index=True)
    if 'uprn' in full.columns and 'uprn_c' not in full.columns:
        full = full.rename(columns={'uprn': 'uprn_c'})
    full['premises_id'] = full['premises_id'].astype(str)
    if 'uprn_c' in full.columns:
        full['uprn_c'] = (full['uprn_c'].astype(str)
                          .str.replace(r'\.0$', '', regex=True))
    return full


print('Matching helpers defined.')

Matching helpers defined.


## 8. Run all-Splink pipeline

Single `SplinkStage` with `match_weight >= 10` threshold.
Every matched row receives a `match_weight` and `distinguishability` score.

In [9]:
STAGES = [
    SplinkStage(
        predict_threshold_match_weight=-20,
        final_match_weight_threshold=10,
        include_full_postcode_block=False,
        retain_intermediate_calculation_columns=False,
    ),
]

print('Running all-Splink pipeline (6 address-key rounds) ...')
print()
results = run_matching_loop(pair_list_6, STAGES, label='Exp2')
results.to_csv('full_result_exp2.csv', index=False)
print(f'\nSaved {len(results):,} rows to full_result_exp2.csv')

matched = results[results['uprn_c'].notna() & (results['uprn_c'] != 'nan')]
hit_premises = matched.groupby('premises_id').first().reset_index()
total = results['premises_id'].nunique()
print(f'Unique premises: {total:,}')
print(f'Matched premises (hit): {len(hit_premises):,} ({len(hit_premises)/total:.1%})')

Running all-Splink pipeline (6 address-key rounds) ...

  [Exp2] Round 0 (method 'a') — 280,329 x 1,052,921
    -> 101,336 matched in 80.5s
  [Exp2] Round 1 (method 'b') — 161,796 x 1,027,922
    -> 138,412 matched in 79.8s
  [Exp2] Round 2 (method 'c') — 161,796 x 1,008,306
    -> 135,633 matched in 70.0s
  [Exp2] Round 3 (method 'd') — 161,796 x 1,062,190
    -> 46,983 matched in 64.1s
  [Exp2] Round 4 (method 'e') — 161,796 x 974,743
    -> 133,796 matched in 58.9s
  [Exp2] Round 5 (method 'f') — 161,796 x 1,051,097
    -> 140,807 matched in 82.9s
  [Exp2] All rounds done in 457.3s

Saved 1,089,309 rows to full_result_exp2.csv
Unique premises: 127,161
Matched premises (hit): 118,464 (93.2%)


## 9. Reformat results

In [10]:
TARGET_COLUMNS = [
    'uid_m', 'uprn_m', 'premises_id', 'address_m', 'uid_c',
    'match_weight', 'distinguishability', 'fuzz_similarity',
    'uprn_c', 'address_c', 'method',
]

exp2_share = results.rename(columns={'uid_ma': 'uid_m', 'uid_ca': 'uid_c'})

drop_cols = [c for c in ['uid_mb', 'uid_cb', 'uid_cc', 'uid_cd', 'uid_ce', 'uid_cf']
             if c in exp2_share.columns]
exp2_share = exp2_share.drop(columns=drop_cols)

final_cols = [c for c in TARGET_COLUMNS if c in exp2_share.columns]
exp2_share = exp2_share[final_cols]

out_file = 'exp2_results_reformatted.csv'
exp2_share.to_csv(out_file, index=False)

print(f'Saved: {out_file}')
print(f'Columns: {list(exp2_share.columns)}')
print(f'Rows: {len(exp2_share):,}')
print(f'\nFirst 5 matched rows:')
print(exp2_share[exp2_share['uprn_c'].notna()].head().to_string(index=False, max_colwidth=40))

Saved: exp2_results_reformatted.csv
Columns: ['uid_m', 'uprn_m', 'premises_id', 'address_m', 'uid_c', 'match_weight', 'distinguishability', 'fuzz_similarity', 'uprn_c', 'address_c', 'method']
Rows: 1,089,309
Column order matches target: True

First 5 matched rows:
uid_m  uprn_m premises_id                                address_m  uid_c  match_weight  distinguishability  fuzz_similarity    uprn_c                                address_c method
    1     NaN    50000000 SAMANTHA CUSICK, WESTBOURNE PARK ROAD... 303117     32.496069           39.128819             93.0 217092023 SAMANTHA CUSICK LONDON, WESTBOURNE PA...      a
    0     NaN    50000000 VACANT PROPERTY, WESTBOURNE PARK ROAD...    NaN           NaN                 NaN              NaN       nan                                      NaN      a
    2     NaN    50000000     10500, WESTBOURNE PARK ROAD, W11 1EH    NaN           NaN                 NaN              NaN       nan                                      NaN      a
   